In [1]:
# Cell 1: Imports, Setup, and Helper Functions
import polars as pl
import numpy as np
from pathlib import Path
import gc
from datetime import datetime
import tempfile
from polars_proc_compare import DataCompare
from polars_proc_compare.data_generator import create_delta_dataset

# Create directories for data and results
data_dir = Path("data")
results_dir = Path("results")
data_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

# Helper functions
def format_number(value):
    """Format numbers with thousands separator and proper decimal places."""
    try:
        num = float(value)
        if num.is_integer():
            return f"{int(num):,}"
        return f"{num:,.2f}"
    except (ValueError, TypeError):
        return str(value)

def create_stat_card(label, value):
    """Create an HTML card displaying a statistic with label."""
    return f"""
    <div class="stat-card">
        <div class="stat-value">{value}</div>
        <div class="stat-label">{label}</div>
    </div>
    """

def create_dataset_summary(base_df, compare_df):
    """Create an HTML summary comparing the two datasets."""
    return f"""
    <div class="dataset-summary">
        <h3>Dataset Summary</h3>
        <div class="dataset-info">
            <div class="dataset-info-cell dataset-info-header"></div>
            <div class="dataset-info-cell dataset-info-header">Observations</div>
            <div class="dataset-info-cell dataset-info-header">Variables</div>
            
            <div class="dataset-info-cell dataset-row-header">Base</div>
            <div class="dataset-info-cell">{format_number(len(base_df))}</div>
            <div class="dataset-info-cell">{format_number(len(base_df.columns))}</div>
            
            <div class="dataset-info-cell dataset-row-header">Compare</div>
            <div class="dataset-info-cell">{format_number(len(compare_df))}</div>
            <div class="dataset-info-cell">{format_number(len(compare_df.columns))}</div>
        </div>
    </div>
    """

def create_variables_summary(results):
    """Create an HTML summary of variable comparisons."""
    return f"""
    <div class="summary-section">
        <h3>Variables Summary</h3>
        <div class="info-grid">
            <div class="info-cell">
                <div class="info-value">{format_number(len(results.structure_results.get('common_cols', [])))}</div>
                <div class="info-label">Variables in Common</div>
            </div>
            <div class="info-cell">
                <div class="info-value">{format_number(len(results.structure_results.get('base_only', [])))}</div>
                <div class="info-label">Variables in Base Only</div>
            </div>
            <div class="info-cell">
                <div class="info-value">{format_number(len(results.structure_results.get('compare_only', [])))}</div>
                <div class="info-label">Variables in Compare Only</div>
            </div>
        </div>
    </div>
    """

def create_observations_summary(results):
    """Create an HTML summary of observation comparisons."""
    return f"""
    <div class="summary-section">
        <h3>Observations Summary</h3>
        <div class="info-grid">
            <div class="info-cell">
                <div class="info-value">{format_number(results.structure_results.get('base_nrows', 0))}</div>
                <div class="info-label">Observations in Base</div>
            </div>
            <div class="info-cell">
                <div class="info-value">{format_number(results.structure_results.get('compare_nrows', 0))}</div>
                <div class="info-label">Observations in Compare</div>
            </div>
            <div class="info-cell">
                <div class="info-value">{format_number(results.structure_results.get('matched_rows', 0))}</div>
                <div class="info-label">Observations in Common</div>
            </div>
            <div class="info-cell">
                <div class="info-value">{format_number(results.structure_results.get('compare_only_rows', 0))}</div>
                <div class="info-label">Observations in Compare Only</div>
            </div>
        </div>
    </div>
    """

def create_values_summary(results):
    """Create an HTML summary of value comparisons."""
    return f"""
    <div class="summary-section">
        <h3>Values Comparison Summary</h3>
        <div class="info-grid">
            <div class="info-cell highlight">
                <div class="info-value">{format_number(results.total_differences)}</div>
                <div class="info-label">Differences Found</div>
            </div>
            <div class="info-cell highlight">
                <div class="info-value">{format_number(len(results.statistics))}</div>
                <div class="info-label">Columns with Differences</div>
            </div>
        </div>
    </div>
    """

def create_column_differences(results, max_observations=25):
    """Create HTML for detailed column-by-column differences."""
    html = []
    
    # Add JavaScript for handling accordion behavior with smooth scrolling
    html.append("""
    <script>
    document.addEventListener('DOMContentLoaded', function() {
        // Get all column sections
        const sections = document.querySelectorAll('.column-diff-section');
        
        // Set first section as active initially
        if (sections.length > 0) {
            sections[0].classList.add('active');
        }
        
        // Add click handlers to all headers
        sections.forEach(section => {
            section.querySelector('.column-header').addEventListener('click', () => {
                // Close all other sections
                sections.forEach(s => {
                    if (s !== section) {
                        s.classList.remove('active');
                    }
                });
                
                // Toggle current section
                const wasActive = section.classList.contains('active');
                section.classList.toggle('active');
                
                // If section is being opened (not closed), scroll it into view
                if (!wasActive) {
                    // Small delay to allow content to expand before scrolling
                    setTimeout(() => {
                        section.scrollIntoView({ 
                            behavior: 'smooth', 
                            block: 'start',
                            inline: 'nearest'
                        });
                    }, 10);
                }
            });
        });
    });
    </script>
    """)
    
    # Rest of the function remains the same...
    
    # Generate HTML for each column
    for col, stats in results.statistics.items():
        # Get column type from variable_types
        col_type = results.structure_results.get('variable_types', {}).get(col, '')
        
        # Create the differences table rows (limited to max_observations)
        diff_rows = []
        total_diffs = stats.get('n_differences', 0)
        for diff in list(stats.get('first_n_differences', []))[:max_observations]:
            diff_rows.append(f"""
                    <tr>
                        <td class="numeric">{diff.get('obs', '')}</td>
                        <td class="numeric">{format_number(diff.get('base', ''))}</td>
                        <td class="numeric">{format_number(diff.get('compare', ''))}</td>
                        <td class="numeric">{format_number(diff.get('abs_diff', ''))}</td>
                        <td class="numeric">{format_number(diff.get('pct_diff', ''))}</td>
                    </tr>""")
        
        # Add a note if there are more differences than shown
        remaining_diffs = total_diffs - len(diff_rows)
        if remaining_diffs > 0:
            diff_rows.append(f"""
                    <tr class="more-differences">
                        <td colspan="5">... and {format_number(remaining_diffs)} more differences</td>
                    </tr>""")
        
        html.append(f"""
        <div class="column-diff-section">
            <div class="column-header">
                <div class="column-name">{col}</div>
                <div class="column-type">Type: {col_type}</div>
            </div>
            <div class="column-content">
                <div class="column-stats">
                    <div class="stat-item">
                        <span class="stat-label">Differences:</span>
                        <span class="stat-value">{format_number(stats.get('n_differences', 0))}</span>
                    </div>
                    {f'''<div class="stat-item">
                        <span class="stat-label">Max Difference:</span>
                        <span class="stat-value">{format_number(stats.get('max_diff', 0))}</span>
                    </div>''' if stats.get('max_diff') is not None else ''}
                    {f'''<div class="stat-item">
                        <span class="stat-label">Mean Difference:</span>
                        <span class="stat-value">{format_number(stats.get('mean_diff', 0))}</span>
                    </div>''' if stats.get('mean_diff') is not None else ''}
                </div>
                <table class="diff-table">
                    <thead>
                        <tr>
                            <th>Observation</th>
                            <th>Base Value</th>
                            <th>Compare Value</th>
                            <th>Difference</th>
                            <th>% Difference</th>
                        </tr>
                    </thead>
                    <tbody>
                        {''.join(diff_rows)}
                    </tbody>
                </table>
            </div>
        </div>
        """)
    
    return '\n'.join(html)

# Professional CSS for HTML report styling
report_css = """
<style>
    body {
        font-family: Arial, sans-serif;
        line-height: 1.6;
        margin: 20px;
        color: #333;
    }

    /* Summary Section */
    .summary {
        background-color: #f8f9fa;
        border: 1px solid #dee2e6;
        border-radius: 4px;
        padding: 20px;
        margin: 20px 0;
        box-shadow: 0 2px 4px rgba(0,0,0,0.1);
    }

    .summary h2 {
        color: #2c3e50;
        margin-top: 0;
        border-bottom: 2px solid #3498db;
        padding-bottom: 10px;
    }

    .summary h3 {
        color: #2c3e50;
        margin-top: 20px;
    }

    /* Tables */
    table {
        width: 100%;
        border-collapse: collapse;
        margin: 25px 0;
        font-size: 14px;
        box-shadow: 0 2px 4px rgba(0,0,0,0.1);
    }

    /* Table Headers */
    th {
        background-color: #3498db;
        color: white;
        font-weight: bold;
        padding: 12px;
        text-align: left;
        border: 1px solid #2980b9;
    }

    /* Table Cells */
    td {
        padding: 10px;
        border: 1px solid #dee2e6;
    }

    /* Numeric Columns */
    td.numeric {
        text-align: right;
        font-family: 'Courier New', monospace;
    }

    /* String Columns */
    td.text {
        text-align: left;
    }

    /* Alternating Rows */
    tr:nth-child(even) {
        background-color: #f8f9fa;
    }

    tr:nth-child(odd) {
        background-color: white;
    }

    /* Row Hover Effect */
    tr:hover {
        background-color: #edf2f7;
    }

    /* Section Headers */
    .section-header {
        background-color: #2c3e50;
        color: white;
        padding: 10px 15px;
        margin: 30px 0 15px 0;
        border-radius: 4px;
    }

    /* Statistics Grid */
    .stats-grid {
        display: grid;
        grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
        gap: 20px;
        margin: 20px 0;
    }

    .stat-card {
        background: white;
        padding: 15px;
        border-radius: 4px;
        border: 1px solid #dee2e6;
        text-align: center;
    }

    .stat-value {
        font-size: 24px;
        font-weight: bold;
        color: #3498db;
    }

    .stat-label {
        color: #666;
        font-size: 14px;
    }

    /* Difference Highlighting */
    .difference {
        color: #e74c3c;
        font-weight: bold;
    }

    /* Dataset Summary Table */
    .dataset-summary {
        margin: 20px 0;
        border: 1px solid #dee2e6;
        border-radius: 4px;
        overflow: hidden;
    }

    .dataset-summary table {
        margin: 0;
    }

    .dataset-summary h3 {
        background-color: #3498db;
        color: white;
        padding: 10px 15px;
        margin: 0;
    }

    .dataset-info {
        display: grid;
        grid-template-columns: auto auto auto;
        gap: 1px;
        background-color: #dee2e6;
    }

    .dataset-info-cell {
        background-color: white;
        padding: 12px 15px;
    }

    .dataset-info-header {
        background-color: #f8f9fa;
        font-weight: bold;
        text-align: center;
    }

    .dataset-row-header {
        background-color: #f8f9fa;
        font-weight: bold;
    }

    /* Summary Sections */
    .summary-section {
        background-color: white;
        border: 1px solid #dee2e6;
        border-radius: 4px;
        margin: 20px 0;
        overflow: hidden;
    }

    .summary-section h3 {
        background-color: #3498db;
        color: white;
        padding: 12px 15px;
        margin: 0;
        font-size: 16px;
    }

    .info-grid {
        display: grid;
        grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
        gap: 1px;
        background-color: #dee2e6;
        padding: 1px;
    }

    .info-cell {
        background-color: white;
        padding: 15px;
        text-align: center;
    }

    .info-value {
        font-size: 20px;
        font-weight: bold;
        color: #2c3e50;
        margin-bottom: 5px;
    }

    .info-label {
        color: #666;
        font-size: 13px;
    }

    .info-cell.highlight .info-value {
        color: #e74c3c;
    }

    /* Column Difference Sections */
    .column-diff-section {
        cursor: pointer;
        scroll-margin-top: 20px;
        margin-bottom: 20px;
    }

    .column-header {
        background: #3498db;
        color: white;
        padding: 12px 15px;
        display: flex;
        justify-content: space-between;
        align-items: center;
        cursor: pointer;
        user-select: none;
        position: sticky;
        top: 20px;
        z-index: 1;
    }

    .column-content {
        display: none;
        background: white;
        transition: all 0.3s ease-in-out;
    }

    .column-stats {
        background: #f8f9fa;
        padding: 15px;  /* Increased padding */
        display: flex;
        gap: 25px;  /* Increased gap between stat items */
        border-bottom: 1px solid #dee2e6;
        margin-top: 1px;  /* Add slight separation from header */
    }

    .stat-item {
        display: flex;
        align-items: center;
        gap: 10px;  /* Increased gap between label and value */
        padding: 5px 0;  /* Added vertical padding */
    }

    .stat-item .stat-label {
        color: #666;
        font-size: 13px;
        font-weight: 500;  /* Made labels slightly bolder */
    }

    .stat-item .stat-value {
        font-weight: bold;
        color: #2c3e50;
        font-size: 16px;  /* Increased font size slightly */
    }

    .diff-table {
        margin: 0 !important;
        box-shadow: none !important;
    }

    .diff-table th {
        background-color: #f8f9fa;
        color: #2c3e50;
        font-weight: bold;
        text-align: right;
    }

    .diff-table th:first-child {
        text-align: left;
    }

    .diff-table td.numeric {
        font-family: 'Courier New', monospace;
        text-align: right;
        padding: 8px 12px;
    }

    .diff-table td:first-child {
        text-align: left;
        font-weight: bold;
    }

    .diff-table tr:hover {
        background-color: #f1f7fe;
    }

    /* More Differences Row */
    .more-differences td {
        text-align: center !important;
        color: #666;
        font-style: italic;
        background-color: #f8f9fa;
        padding: 8px !important;
    }

    /* Comparison Results Section */
    .comparison-results {
        margin-top: 30px;
    }

    .comparison-results h2 {
        background-color: #2c3e50;
        color: white;
        padding: 12px 15px;
        margin: 0 0 20px 0;  /* Added bottom margin */
        border-radius: 4px 4px 0 0;
    }
    
    /* Collapsible Sections */
    .column-diff-section {
        cursor: pointer;
        scroll-margin-top: 20px;
        margin-bottom: 20px;
    }

    .column-content {
        display: none;
        background: white;
    }

    .column-diff-section.active .column-content {
        display: block;
    }

    .column-header {
        cursor: pointer;
        user-select: none;
    }

    .column-header:hover {
        background: #2980b9;
    }

    /* Collapsible Sections */
    .column-diff-section {
        cursor: pointer;
        scroll-margin-top: 20px;  /* Adds some space at the top when scrolled */
    }

    .column-content {
        display: none;
        background: white;
        transition: all 0.3s ease-in-out;
    }

    .column-diff-section.active .column-content {
        display: block;
    }

    .column-header {
        cursor: pointer;
        user-select: none;
        position: sticky;
        top: 20px;  /* Keeps header visible while scrolling through content */
        z-index: 1;
    }
</style>
"""

In [2]:
# Cell 2: Generate Base Dataset
def create_large_dataset(n_rows: int, n_cols: int, seed: int = 42) -> pl.DataFrame:
    """Create a large dataset with various data types."""
    np.random.seed(seed)
    
    # Create timestamps using Python's datetime
    start_ts = datetime(2024, 1, 1).timestamp()
    end_ts = datetime(2024, 12, 31).timestamp()
    timestamps = np.linspace(start_ts, end_ts, n_rows)
    
    # Create base data with explicit types
    data = {
        "id": pl.Series("id", range(n_rows), dtype=pl.Int64),  # Use Int64 instead of Int32
        "timestamp": pl.Series(
            "timestamp",
            timestamps,
            dtype=pl.Int64
        ).cast(pl.Datetime("ms"))
    }
    
    # Add numeric columns with non-optimal types
    for i in range(n_cols):
        if i % 3 == 0:  # Integer columns
            data[f"int_col_{i}"] = pl.Series(
                name=f"int_col_{i}",
                values=np.random.randint(-10000, 10000, n_rows),
                dtype=pl.Int64  # Use Int64 instead of Int32
            )
        elif i % 3 == 1:  # Float columns
            data[f"float_col_{i}"] = pl.Series(
                name=f"float_col_{i}",
                values=np.random.normal(0, 100, n_rows),
                dtype=pl.Float64  # Use Float64 instead of Float32
            )
        else:  # String columns (instead of categorical)
            categories = [f"cat_{j}" for j in range(10)]
            values = np.random.choice(categories, n_rows)
            data[f"cat_col_{i}"] = pl.Series(
                name=f"cat_col_{i}",
                values=values,
                dtype=pl.Utf8  # Use Utf8 instead of Categorical
            )
        
        # Free up memory after each column
        if i % 5 == 0:
            _ = gc.collect()
    
    df = pl.DataFrame(data)
    print(f"DataFrame schema: {df.schema}")
    return df

try:
    # Start with a very small dataset for testing
    print("Creating base dataset...")
    base_df = create_large_dataset(n_rows=10_000, n_cols=5)
    print(f"Base dataset shape: {base_df.shape}")
    print(f"Base dataset schema:\n{base_df.schema}")
    
    # Save to parquet
    base_path = data_dir / "base_large.parquet"
    base_df.write_parquet(base_path)
    print(f"Base dataset saved to {base_path}")
    
except Exception as e:
    print(f"Error: {str(e)}")
    import traceback
    traceback.print_exc()
    # Clean up memory
    if 'base_df' in locals(): del base_df
    _ = gc.collect()

Creating base dataset...
DataFrame schema: OrderedDict({'id': Int64, 'timestamp': Datetime(time_unit='ms', time_zone=None), 'int_col_0': Int64, 'float_col_1': Float64, 'cat_col_2': Utf8, 'int_col_3': Int64, 'float_col_4': Float64})
Base dataset shape: (10000, 7)
Base dataset schema:
OrderedDict({'id': Int64, 'timestamp': Datetime(time_unit='ms', time_zone=None), 'int_col_0': Int64, 'float_col_1': Float64, 'cat_col_2': Utf8, 'int_col_3': Int64, 'float_col_4': Float64})
Base dataset saved to data/base_large.parquet


In [3]:
# Cell 3: Generate Delta Dataset with Detailed Statistics
print("Creating comparison dataset with modifications...")

# Create delta dataset with 5% differences
compare_df, modifications = create_delta_dataset(
    base_df,
    delta_percentage=5.0,  # 5% differences
    seed=42,
    exclude_columns=["id", "timestamp"]
)

print("\nDelta Injection Statistics:")
print(f"Total rows: {modifications['total_rows']:,}")
print(f"Total columns: {modifications['total_columns']}")
print(f"Total cells: {modifications['total_cells']:,}")
print(f"Modified cells: {modifications['modified_cells']:,}")
print(f"Modified rows: {modifications['modified_rows']:,}")
print("\nModifications by column:")
for col, count in modifications['modified_columns'].items():
    print(f"- {col}: {count:,} changes ({count/modifications['total_rows']*100:.1f}% of rows)")

# Save comparison dataset
compare_path = data_dir / "compare_data.parquet"
compare_df.write_parquet(compare_path)
print(f"\nComparison dataset saved to {compare_path}")

Creating comparison dataset with modifications...

Delta Injection Statistics:
Total rows: 10,000
Total columns: 5
Total cells: 50,000
Modified cells: 2,500
Modified rows: 2,266

Modifications by column:
- int_col_0: 500 changes (5.0% of rows)
- float_col_1: 500 changes (5.0% of rows)
- cat_col_2: 500 changes (5.0% of rows)
- int_col_3: 500 changes (5.0% of rows)
- float_col_4: 500 changes (5.0% of rows)

Comparison dataset saved to data/compare_data.parquet


In [4]:
# Cell 4: Run In-Memory Comparison with HTML Report
print("Running in-memory comparison...")

# Run comparison
dc = DataCompare(base_df, compare_df, key_columns=["id"])
results = dc.compare()

# Generate HTML report with professional styling
html_header = f"""
<!DOCTYPE html>
<html>
<head>
{report_css}
</head>
<body>
    <h1 class="section-header">Comparison Report - In-Memory Mode</h1>
    
    {create_dataset_summary(base_df, compare_df)}
    {create_variables_summary(results)}
    {create_observations_summary(results)}
    {create_values_summary(results)}

    <div class="comparison-results">
        <h2>Detailed Comparison Results</h2>
        {create_column_differences(results, max_observations=25)}
    </div>
</body>
</html>
"""

# Save reports
in_memory_html = results_dir / "in_memory_report.html"
in_memory_csv = results_dir / "in_memory_differences.csv"

with open(in_memory_html, "w") as f:
    f.write(html_header)

results.to_csv(in_memory_csv)
print(f"\nReports generated:")
print(f"- HTML report: {in_memory_html}")
print(f"- CSV differences: {in_memory_csv}")

Running in-memory comparison...

Reports generated:
- HTML report: results/in_memory_report.html
- CSV differences: results/in_memory_differences.csv


In [5]:
# Cell 5: Run Disk-Based Comparison with Memory Optimization
print("Running disk-based comparison...")

with tempfile.TemporaryDirectory() as temp_dir:
    # Use disk-based comparison with memory optimization
    dc = DataCompare(
        base_df=str(base_path),
        compare_df=str(compare_path),
        key_columns=["id"],
        disk_mode=True,
        optimize_dtypes=True,
        max_memory_usage=1024,
        temp_dir=temp_dir
    )

    results = dc.compare()

    # Generate HTML report with professional styling
    html_header = f"""
    <!DOCTYPE html>
    <html>
    <head>
    {report_css}
    </head>
    <body>
        <h1 class="section-header">Comparison Report - Disk-Based Mode</h1>
        
        {create_dataset_summary(base_df, compare_df)}
        {create_variables_summary(results)}
        {create_observations_summary(results)}
        {create_values_summary(results)}

        <div class="comparison-results">
            <h2>Detailed Comparison Results</h2>
            {create_column_differences(results, max_observations=25)}
        </div>
    </body>
    </html>
    """

    # Save reports
    disk_html = results_dir / "disk_based_report.html"
    disk_csv = results_dir / "disk_based_differences.csv"

    with open(disk_html, "w") as f:
        f.write(html_header)

    results.to_csv(disk_csv)
    print(f"\nReports generated:")
    print(f"- HTML report: {disk_html}")
    print(f"- CSV differences: {disk_csv}")

Running disk-based comparison...

Original schema:
__row_id: UInt32
id: Int64
timestamp: Datetime(time_unit='ms', time_zone=None)
int_col_0: Int64
float_col_1: Float64
cat_col_2: Utf8
int_col_3: Int64
float_col_4: Float64
Optimized id: Int64 -> UInt16
Optimized int_col_0: Int64 -> Int16
Optimized float_col_1: Float64 -> Float32
String column stats: 10 unique values out of 10000 (0.10%)
Converting to categorical (ratio 0.10% <= threshold 50.00%)
Optimized cat_col_2: Utf8 -> Categorical
Optimized int_col_3: Int64 -> Int16
Optimized float_col_4: Float64 -> Float32

Original schema:
__row_id: UInt32
id: Int64
timestamp: Datetime(time_unit='ms', time_zone=None)
int_col_0: Int64
float_col_1: Float64
cat_col_2: Utf8
int_col_3: Int64
float_col_4: Float64
Optimized id: Int64 -> UInt16
Optimized int_col_0: Int64 -> Int16
Optimized float_col_1: Float64 -> Float32
String column stats: 20 unique values out of 10000 (0.20%)
Converting to categorical (ratio 0.20% <= threshold 50.00%)
Optimized cat_co